# Stage 2: adaptive selected-pair-clean maximum repeat count

Only Stage-1-compatible modules enter this experiment. Each exact GA DNA is scored by the live IDT API; all finite rule Scores are summed and only totals strictly below 10 pass. Positive-score rules reweight the next GA attempt at the same copy count.

In [ ]:
RESULTS = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/expanded-middle-repeatsdb-foldseek-v1/maximum_copy_results.parquet'
TRACE = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/expanded-middle-repeatsdb-foldseek-v1/adaptive_copy_search_trace.parquet'
FINAL_SUMMARY = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/tables/expanded-middle-repeatsdb-foldseek-v1/module_final_summary.parquet'
FIGURE_STEM = '/home/wendai/projects/hurdler/clone_repeat_protein/studies/hurdler_validation/step04_module_optimization/figures/expanded-middle-repeatsdb-foldseek-v1/module_length_vs_maximum_copies'

In [ ]:
from pathlib import Path
import hashlib, json
import pandas as pd
VERSION = 'expanded-middle-repeatsdb-foldseek-v1'
def sha256(path):
    path = Path(path)
    if not path.is_file(): return None
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
def read_optional(path):
    path = Path(path)
    if not path.is_file(): return None
    return pd.read_parquet(path) if path.suffix == '.parquet' else pd.read_csv(path)
run_context = {'corpus_version': VERSION, 'rules_version': 'legacy-optimized-v1', 'inputs': {}, 'row_counts': {}, 'filter_flow': [], 'limitations': [], 'status': 'passed'}

In [ ]:
from hurdler.module_experiments import plot_maximum_copy_scatter
import pyarrow.parquet as pq
results=read_optional(RESULTS); final_summary=read_optional(FINAL_SUMMARY)
trace_rows=pq.ParquetFile(TRACE).metadata.num_rows if Path(TRACE).is_file() else None
for path in (RESULTS,TRACE,FINAL_SUMMARY): run_context['inputs'][Path(path).name]=sha256(path)
if results is None or trace_rows is None or final_summary is None:
    run_context['status']='production_pending'; run_context['limitations'].append('Adaptive GA/IDT production shards are incomplete; no maximum is imputed as zero or one.')
    outcomes=pd.DataFrame({'status':['production_pending']})
else:
    accepted=results.final_passed.fillna(False)
    assert pd.to_numeric(results.loc[accepted,'verified_max_copies']).ge(2).all()
    assert results.loc[accepted,'selected_pair_re_site_excess'].eq(0).all()
    assert pd.to_numeric(results.loc[accepted,'idt_complexity_score']).lt(10).all()
    assert results.loc[accepted,'idt_response_sha256'].astype(str).str.len().eq(64).all()
    assert results.loc[accepted,'validation_passed'].fillna(False).all()
    proof=results.loc[accepted,'adaptive_maximum_proof_status'].isin(['capacity_limit_reached','next_copy_failed_at_100'])
    assert proof.all()
    outcomes=results.groupby(['collection','fragment_limit_bp','final_status']).size().rename('modules').reset_index()
    plot_maximum_copy_scatter(results,FIGURE_STEM)
run_context['row_counts']={'compatible_capacity_rows':None if results is None else len(results),'GA_IDT_attempts':trace_rows,'final_summary_rows':None if final_summary is None else len(final_summary),'verified_maxima':None if results is None else int(accepted.sum())}
outcomes

In [ ]:
if trace_rows is not None:
    import duckdb
    escaped_trace=str(TRACE).replace("'","''")
    feedback=duckdb.connect().execute(f"SELECT copies, generations, count(*) AS rejected_attempts FROM read_parquet('{escaped_trace}') WHERE try_cast(idt_complexity_score AS DOUBLE) >= 10 GROUP BY copies, generations ORDER BY copies, generations").fetchdf()
else: feedback=pd.DataFrame({'status':['production_pending']})
feedback.head(30)

In [ ]:
from IPython.display import Image, display
figure=Path(str(FIGURE_STEM)+'.png')
if figure.is_file(): display(Image(filename=str(figure)))
else: print('Scatter pending finalized Stage-2 tables:', figure)

In [ ]:
run_context['filter_flow']=['freeze each Stage-1 selected plasmid and Site-I/Site-II pair', 'require exact translation of middle_unit × n', 'hard constraint: zero excess selected-pair sites', 'soft scores: Site III, nonselected RE sites, GC, repeats, hairpins, CAI', '10-generation binary lower-bound search', 'advance one copy at a time with 10/20/40/60/80/100 generations', 'score every completed GA DNA through live IDT API', 'sum every finite rule Score and accept only sum <10', 'use positive rule names/reasons to raise mapped GA weights before same-length retry', 'report >=2 only after cap or next-copy 100-generation proof']
run_context['limitations'].append('IDT API failures are unclassified and cannot pass; credentials and raw submitted DNA are never stored in Git.')

In [ ]:
run_context